# Open-weight VLM roster benchmark (Kaggle, 2x T4)

Runs **7 locally-loaded open-weight VLMs** over the same V4a (synthetic tag) and V4b (real
chest tube) crops, under the *exact same blind prompt*, one model at a time with full VRAM
cleanup between models.

**Roster** (medical specialists first, then general VLMs):
1. `StanfordAIMI/CheXagent-2-3b`
2. `StanfordAIMI/CheXagent-8b`
3. `google/medgemma-4b-it`
4. `Qwen/Qwen2.5-VL-3B-Instruct`
5. `Qwen/Qwen2.5-VL-7B-Instruct`  <- size axis vs the 3B, same family, everything else held constant
6. `microsoft/Phi-3.5-vision-instruct`
7. `HuggingFaceTB/SmolVLM2-2.2B-Instruct`

## Kaggle setup -- REQUIRED before running

1. **Accelerator: `GPU T4 x2`** (Settings sidebar). Two 16GB cards, ~31GB via sharding --
   NOT one 32GB pool. Small models land entirely on GPU 0; only the big ones shard.
2. **Internet: ON** (Settings sidebar). Off by default on Kaggle. Without it, every model
   download fails and the whole roster reports FAILED.
3. **Add your crops as a Kaggle Dataset** containing these four files:
   `crops_v4a.zip`, `manifest_v4a.json`, `crops_v4b.zip`, `manifest_v4b.json`.
   The dataset can be named anything -- the resolver below searches `/kaggle/input`
   recursively rather than hardcoding a slug. Note Kaggle **auto-extracts zips at upload
   time**, so the crops may arrive as folders instead of `.zip` files; the resolver handles
   both.
4. **HF token** (only for the gated `google/medgemma-4b-it`): Add-ons -> Secrets, name it
   `HF_TOKEN`. Accept the license at https://huggingface.co/google/medgemma-4b-it first.
   Skip this and medgemma alone fails; the other six still run.

## Notes carried over from the failed 2026-07-16 Colab run

- `transformers` is pinned `>=4.50,<5` at **both** ends. Unbounded `>=4.50` resolved to
  5.14.0, which removed the TF/Flax backend that CheXagent's `trust_remote_code` code imports.
- **Pillow is not touched.** `-U pillow` left it internally inconsistent and broke the
  torchvision import chain, taking out 5 of 6 models with an error naming no model at all.
- **dtype is auto-detected, not hardcoded.** bf16 needs compute capability >= 8.0 (Ampere);
  T4 is 7.5 and P100 is 6.0, so this run will select fp16. The earlier notebook hardcoded
  bf16 -- a latent bug that never surfaced only because everything died at import first.

## Output

Inference only. Writes `/kaggle/working/outputs/predictions_<model>_<dataset>.jsonl`
(`{"id", "kind", "description"}` per line) -- the schema
`src/auditor/discovery/score_naming.py` already reads. Score locally:

```
python -m auditor.discovery score --manifest manifest_v4a.json \
    --predictions predictions_<model>_v4a.jsonl --confound tag
python -m auditor.discovery score --manifest manifest_v4b.json \
    --predictions predictions_<model>_v4b.jsonl --confound tube
```

## 1. Installation


In [ ]:
# See the intro: every constraint here traces to a real failure on 2026-07-16.
#   <5      -- 5.x removed is_tf_available, which CheXagent's remote code imports
#   >=4.50  -- MedGemma (Gemma 3) requires it
#   no pillow/protobuf/sentencepiece -- Kaggle ships them; upgrading broke Pillow
#   no blanket -U -- that is what caused both failures above
#   torch/torchvision unpinned -- Kaggle ships CUDA-linked builds
!pip -q install "transformers>=4.50,<5" accelerate bitsandbytes qwen-vl-utils einops


### Environment guard

Fails in seconds if the install left a broken environment, instead of after ~25GB of model
downloads and seven identical tracebacks.

In [ ]:
# Verify the environment BEFORE any model download. Each check maps to a real failure.
import PIL, transformers

major = int(transformers.__version__.split(".")[0])
assert major == 4, (
    f"transformers is {transformers.__version__}, expected 4.x. The <5 bound did not take "
    "effect -- most likely transformers was already imported in this session, so pip's "
    "change never loaded. Run -> Restart & clear cell outputs, then re-run from cell 1."
)

# PIL must be importable THROUGH torchvision, which transformers.image_utils pulls in.
# A half-upgraded Pillow passes a bare `import PIL` but dies here.
import torchvision                       # noqa: F401  -- the import IS the test
from transformers import AutoProcessor   # noqa: F401  -- pulls the image_utils chain

print(f"OK | transformers {transformers.__version__} | pillow {PIL.__version__} | "
      f"torchvision imports cleanly")


## 2. Imports

In [ ]:
from __future__ import annotations

import gc
import json
import time
import traceback
import zipfile
from pathlib import Path
from typing import Callable

import torch
from PIL import Image
from tqdm.auto import tqdm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
assert DEVICE == "cuda", "No GPU. Settings -> Accelerator -> GPU T4 x2, then re-run."

N_GPUS = torch.cuda.device_count()
for i in range(N_GPUS):
    p = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {p.name} | {p.total_memory / 1e9:.1f} GB | compute capability "
          f"{p.major}.{p.minor}")

# dtype auto-detect. bf16 needs compute capability >= 8.0 (Ampere); T4 is 7.5 -> fp16.
# Detected rather than hardcoded so this notebook stays correct if run on an Ampere card.
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(f"\n{N_GPUS} GPU(s) | bf16 supported: {torch.cuda.is_bf16_supported()} -> using {DTYPE}")
if N_GPUS < 2:
    print("NOTE: only 1 GPU visible. Everything still runs; CheXagent-8b at fp16 (~16GB) is "
          "the one model that may not fit without sharding across two cards.")


## 3. Hugging Face auth (gated `google/medgemma-4b-it` only)

Reads a Kaggle Secret named `HF_TOKEN`. If it is missing, only medgemma fails-and-skips --
the other six models are unaffected.

In [ ]:
def hf_login() -> bool:
    """Log in to HF from the Kaggle Secret `HF_TOKEN`. Returns True on success."""
    try:
        from huggingface_hub import login
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("HF_TOKEN")
        login(token=token)
        print("HF login OK -- gated models available")
        return True
    except Exception as e:
        print(f"HF login unavailable ({type(e).__name__}: {e})")
        print("  -> google/medgemma-4b-it will fail-and-skip. All other models unaffected.")
        print("  -> To fix: Add-ons -> Secrets -> add HF_TOKEN, and accept the license at")
        print("     https://huggingface.co/google/medgemma-4b-it")
        return False

HF_OK = hf_login()


## 4. Locate Dataset

`/kaggle/input` is **read-only**, so any zip is extracted into `/kaggle/working`. The
resolver searches recursively instead of hardcoding a dataset slug, and handles Kaggle
auto-extracting zips at upload time (crops may arrive as a folder, not a `.zip`).

In [ ]:
INPUT_DIR = Path("/kaggle/input")
WORK_DIR = Path("/kaggle/working")


def find_one(filename: str) -> Path | None:
    """Find a single file by name anywhere under /kaggle/input."""
    hits = sorted(INPUT_DIR.rglob(filename))
    return hits[0] if hits else None


DATASET_TOKENS = ("v4a", "v4b")


def _claims_dataset(path: Path, dataset: str) -> bool:
    """True if `path` belongs to `dataset`, judged by its DEEPEST unambiguous component.

    Why not a plain `dataset in str(path)`: the /kaggle/input/<slug>/ component is a
    user-chosen dataset name, and a natural one like "vlm-v4a-v4b-crops" mentions BOTH
    datasets -- so a whole-path substring test matches every candidate for every dataset and
    silently returns the wrong crops. That is unrecoverable in practice: v4a and v4b use
    IDENTICAL filenames (pos_000.png ...) and both have exactly 200 rows, so a swap passes
    every downstream check and just quietly scores the wrong images. Verified as a real
    failure before this fix existed.

    So: walk components deepest-first and let the first one naming exactly ONE dataset
    decide. A leaf like crops_v4b/ then beats an ambiguous parent slug.
    """
    parts = str(path).lower().replace("\\", "/").split("/")
    for part in reversed(parts):
        names = [t for t in DATASET_TOKENS if t in part]
        if len(names) == 1:
            return names[0] == dataset
        # len 0 -> uninformative (e.g. "crops"), keep walking up
        # len 2 -> ambiguous (e.g. the "v4a-v4b" slug), keep walking up
    return False


def resolve_crops_dir(dataset: str) -> Path:
    """Return the directory holding <id>.png crops for `dataset` ('v4a' or 'v4b').

    Handles the layouts Kaggle actually produces -- it auto-extracts archives at dataset
    creation, so what you uploaded as a .zip may already be a folder:
      (a) an extracted dir holding *.png directly
      (b) the same, with the pngs one level down in a nested crops/ folder
      (c) a still-zipped crops_<dataset>.zip, extracted here into /kaggle/working

    Ambiguity is a hard error, never a guess -- see _claims_dataset for why a wrong pick
    here would be silent and fatal to the experiment.
    """
    # (a) / (b) -- already extracted somewhere under /kaggle/input
    candidates = [
        d for d in sorted(INPUT_DIR.rglob("*"))
        if d.is_dir() and any(d.glob("*.png")) and _claims_dataset(d, dataset)
    ]
    if len(candidates) > 1:
        raise RuntimeError(
            f"Ambiguous crops for {dataset} -- {len(candidates)} candidate dirs:\n  "
            + "\n  ".join(str(c) for c in candidates)
            + f"\nRefusing to guess: picking wrong would silently score {dataset} against "
              "the other dataset's images. Keep exactly one crops dir per dataset."
        )
    if candidates:
        return candidates[0]

    # (c) -- a zip we must extract into the writable working dir (/kaggle/input is read-only)
    z = find_one(f"crops_{dataset}.zip")
    if z is None:
        raise FileNotFoundError(
            f"No crops found for {dataset}. Expected crops_{dataset}.zip, or an extracted "
            f"folder identifiable as '{dataset}', under /kaggle/input. Did you attach the "
            f"dataset (right sidebar -> Add Input)?"
        )
    dest = WORK_DIR / f"crops_{dataset}"
    dest.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(z) as zf:
        zf.extractall(dest)
    if not any(dest.glob("*.png")):
        nested = dest / "crops"
        if nested.is_dir() and any(nested.glob("*.png")):
            return nested
    return dest


def load_manifest(dataset: str) -> list[dict]:
    p = find_one(f"manifest_{dataset}.json")
    if p is None:
        raise FileNotFoundError(f"manifest_{dataset}.json not found under /kaggle/input")
    with open(p, encoding="utf-8") as f:
        return json.load(f)


DATASETS: dict[str, dict] = {}
for ds in ("v4a", "v4b"):
    crops_dir = resolve_crops_dir(ds)
    manifest = load_manifest(ds)
    n_pos = sum(1 for r in manifest if r["kind"] == "positive")
    n_ctrl = sum(1 for r in manifest if r["kind"] == "control")
    # Fail now if a manifest row has no image -- not 200 times inside the inference loop.
    missing = [r["id"] for r in manifest if not (crops_dir / f"{r['id']}.png").exists()]
    assert not missing, f"{ds}: {len(missing)} manifest rows have no image, e.g. {missing[:5]}"
    DATASETS[ds] = {"manifest": manifest, "crops_dir": str(crops_dir)}
    print(f"{ds}: {len(manifest)} rows ({n_pos} pos, {n_ctrl} ctrl) | all images present")
    print(f"     crops -> {crops_dir}")

# --- Falsifying control on the resolver itself -----------------------------------------
# A v4a/v4b mix-up is the one error nothing downstream could catch: identical filenames,
# identical row counts, so a swap produces plausible numbers for the wrong images. Prove
# they are distinct rather than trusting the path logic.
paths = {ds: d["crops_dir"] for ds, d in DATASETS.items()}
assert len(set(paths.values())) == len(paths), f"v4a and v4b resolved to the SAME dir: {paths}"

# Distinct paths are not proof of distinct CONTENT -- hash the actual bytes.
import hashlib


def _crops_digest(ds: str, n: int = 20) -> str:
    h = hashlib.sha256()
    for p in sorted(Path(DATASETS[ds]["crops_dir"]).glob("*.png"))[:n]:
        h.update(p.read_bytes())
    return h.hexdigest()[:12]


_da, _db = _crops_digest("v4a"), _crops_digest("v4b")
assert _da != _db, (
    f"v4a and v4b crops are BYTE-IDENTICAL ({_da}) -- the same images resolved twice. "
    "Every v4b number would silently be a v4a number. Check the dataset layout."
)
print(f"\nresolver control PASSED -- v4a/v4b are distinct (digest {_da} vs {_db})")


## 5. Prompt

Byte-identical across every model and dataset -- the only independent variable is the model. Matches `BLIND_PROMPT` in `src/auditor/discovery/name_groq.py` so this roster stays comparable to the scout/pixtral/gemini numbers already recorded.

In [ ]:
PROMPT = (
    "This is a small region that a diagnostic model focused on within a chest X-ray. "
    "Describe anything artificial, non-anatomical, or out of place that you see in this "
    "region. If everything looks like normal anatomy, say so. Be specific and concise."
)
MAX_NEW_TOKENS = 64


## 6. Benchmark Functions

Each model family gets a **loader** `(model_id, quantize) -> (proc_or_tok, model)` and a
**generate fn** `(proc_or_tok, model, image_path, prompt) -> str`. Every calling convention
below was verified against that model's own live HF model card on 2026-07-16 -- they differ
more than you would expect, which is why there is no single generic path.

In [ ]:
def make_bnb_config():
    """4-bit config. compute dtype follows the detected DTYPE (fp16 on T4)."""
    from transformers import BitsAndBytesConfig
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=DTYPE,
        bnb_4bit_quant_type="nf4",
    )


def vram_report(label: str) -> None:
    """Print allocated/reserved VRAM for every visible GPU."""
    parts = []
    for i in range(N_GPUS):
        alloc = torch.cuda.memory_allocated(i) / 1e9
        resv = torch.cuda.memory_reserved(i) / 1e9
        parts.append(f"gpu{i} alloc={alloc:.2f}GB resv={resv:.2f}GB")
    print(f"[{label}] " + " | ".join(parts))


def unload(*objs) -> None:
    """Fully release model/processor refs and clear VRAM on every GPU."""
    for o in objs:
        del o
    gc.collect()
    for i in range(N_GPUS):
        with torch.cuda.device(i):
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()


In [ ]:
# --- Generic family: AutoProcessor + AutoModelForImageTextToText ----------------------
# Covers MedGemma and SmolVLM2 (both model cards document this exact path).

def load_generic_itt(model_id: str, quantize: bool):
    from transformers import AutoModelForImageTextToText, AutoProcessor
    processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
    kwargs = dict(device_map="auto", trust_remote_code=True)
    if quantize:
        try:
            kwargs["quantization_config"] = make_bnb_config()
            return processor, AutoModelForImageTextToText.from_pretrained(model_id, **kwargs)
        except Exception as e:
            print(f"  4-bit load failed ({e!r}), falling back to {DTYPE}")
            kwargs.pop("quantization_config", None)
    kwargs["torch_dtype"] = DTYPE
    return processor, AutoModelForImageTextToText.from_pretrained(model_id, **kwargs)


def generate_generic_itt(processor, model, image_path: str, prompt: str) -> str:
    image = Image.open(image_path).convert("RGB")
    messages = [{"role": "user", "content": [{"type": "image", "image": image},
                                              {"type": "text", "text": prompt}]}]
    inputs = processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt",
    ).to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    new_tokens = output[:, inputs["input_ids"].shape[1]:]
    return processor.batch_decode(new_tokens, skip_special_tokens=True)[0].strip()


In [ ]:
# --- Qwen2.5-VL (3B and 7B): its OWN model class + qwen-vl-utils for vision inputs -----
# The card loads Qwen2_5_VLForConditionalGeneration explicitly, not the generic Auto class,
# and uses process_vision_info to build image inputs. Shared by both sizes -- the 3B/7B pair
# is the size axis, so they MUST run through identical code for the comparison to mean
# anything.

def load_qwen25vl(model_id: str, quantize: bool):
    from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration
    processor = AutoProcessor.from_pretrained(model_id)
    kwargs = dict(device_map="auto")
    if quantize:
        try:
            kwargs["quantization_config"] = make_bnb_config()
            return processor, Qwen2_5_VLForConditionalGeneration.from_pretrained(model_id, **kwargs)
        except Exception as e:
            print(f"  4-bit load failed ({e!r}), falling back to {DTYPE}")
            kwargs.pop("quantization_config", None)
    kwargs["torch_dtype"] = DTYPE
    return processor, Qwen2_5_VLForConditionalGeneration.from_pretrained(model_id, **kwargs)


def generate_qwen25vl(processor, model, image_path: str, prompt: str) -> str:
    from qwen_vl_utils import process_vision_info
    messages = [{"role": "user", "content": [{"type": "image", "image": image_path},
                                              {"type": "text", "text": prompt}]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(text=[text], images=image_inputs, videos=video_inputs,
                        padding=True, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    new_tokens = output[:, inputs["input_ids"].shape[1]:]
    return processor.batch_decode(new_tokens, skip_special_tokens=True)[0].strip()


In [ ]:
# --- Phi-3.5-vision-instruct: AutoModelForCausalLM + its own <|image_1|> placeholder ---
# _attn_implementation='eager' because flash_attn is not installed here (the card's default
# is flash_attention_2, which would raise).

def load_phi35_vision(model_id: str, quantize: bool):
    from transformers import AutoModelForCausalLM, AutoProcessor
    processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True, num_crops=4)
    kwargs = dict(device_map="auto", trust_remote_code=True, _attn_implementation="eager")
    if quantize:
        try:
            kwargs["quantization_config"] = make_bnb_config()
            return processor, AutoModelForCausalLM.from_pretrained(model_id, **kwargs)
        except Exception as e:
            print(f"  4-bit load failed ({e!r}), falling back to {DTYPE}")
            kwargs.pop("quantization_config", None)
    kwargs["torch_dtype"] = DTYPE
    return processor, AutoModelForCausalLM.from_pretrained(model_id, **kwargs)


def generate_phi35_vision(processor, model, image_path: str, prompt: str) -> str:
    image = Image.open(image_path).convert("RGB")
    messages = [{"role": "user", "content": f"<|image_1|>\n{prompt}"}]
    text = processor.tokenizer.apply_chat_template(messages, tokenize=False,
                                                    add_generation_prompt=True)
    inputs = processor(text, [image], return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                                 eos_token_id=processor.tokenizer.eos_token_id)
    new_tokens = output[:, inputs["input_ids"].shape[1]:]
    return processor.batch_decode(new_tokens, skip_special_tokens=True)[0].strip()


In [ ]:
# --- CheXagent-2-3b: AutoTokenizer + from_list_format + chat template -----------------
# Verified against the live card 2026-07-16; matches the existing chexagent_naming.ipynb.
# NOT quantized -- trust_remote_code architectures often do not take bitsandbytes cleanly,
# and at 3B it fits a single T4 in fp16 regardless.

def load_chexagent_2_3b(model_id: str, quantize: bool):
    from transformers import AutoModelForCausalLM, AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto",
                                                  trust_remote_code=True)
    model = model.to(DTYPE)
    model.eval()
    return tokenizer, model


def generate_chexagent_2_3b(tokenizer, model, image_path: str, prompt: str) -> str:
    query = tokenizer.from_list_format([{"image": image_path}, {"text": prompt}])
    conv = [{"from": "system", "value": "You are a helpful assistant."},
            {"from": "human", "value": query}]
    input_ids = tokenizer.apply_chat_template(conv, add_generation_prompt=True,
                                               return_tensors="pt")
    output = model.generate(
        input_ids.to(model.device), do_sample=False, num_beams=1, temperature=1.0, top_p=1.0,
        use_cache=True, max_new_tokens=MAX_NEW_TOKENS,
    )[0]
    return tokenizer.decode(output[input_ids.size(1):-1]).strip()


# --- CheXagent-8b: a DIFFERENT convention from the 3b -- AutoProcessor + GenerationConfig
# and a " USER: <s>{prompt} ASSISTANT: <s>" prompt format. Verified against its own live card
# 2026-07-16 after initially (wrongly) assuming it shared the 3b's interface.
#
# 8B at fp16 is ~16GB, over a single T4's 15.6GB usable -- so 4-bit is tried first, and the
# fp16 fallback relies on device_map='auto' sharding across both T4s. UNTESTED either way:
# on 2026-07-16 this model died at import (the Pillow bug) long before memory mattered.

def load_chexagent_8b(model_id: str, quantize: bool):
    from transformers import AutoModelForCausalLM, AutoProcessor, GenerationConfig
    processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
    generation_config = GenerationConfig.from_pretrained(model_id)
    kwargs = dict(trust_remote_code=True, device_map="auto")
    if quantize:
        try:
            kwargs["quantization_config"] = make_bnb_config()
            model = AutoModelForCausalLM.from_pretrained(model_id, **kwargs)
            return (processor, generation_config), model
        except Exception as e:
            print(f"  4-bit load failed ({e!r}), falling back to {DTYPE} across {N_GPUS} GPU(s)")
            kwargs.pop("quantization_config", None)
    kwargs["torch_dtype"] = DTYPE
    model = AutoModelForCausalLM.from_pretrained(model_id, **kwargs)
    return (processor, generation_config), model


def generate_chexagent_8b(proc_and_genconfig, model, image_path: str, prompt: str) -> str:
    processor, generation_config = proc_and_genconfig
    image = Image.open(image_path).convert("RGB")
    inputs = processor(images=[image], text=f" USER: <s>{prompt} ASSISTANT: <s>",
                        return_tensors="pt").to(device=model.device, dtype=DTYPE)
    output = model.generate(**inputs, generation_config=generation_config)[0]
    return processor.tokenizer.decode(output, skip_special_tokens=True).strip()


In [ ]:
# --- Roster -------------------------------------------------------------------------
# Order: medical specialists first, then general. The Qwen 3B/7B pair is the size axis --
# identical code path, identical prompt, identical crops; only parameter count differs.
ROSTER: list[dict] = [
    {"name": "chexagent_2_3b", "model_id": "StanfordAIMI/CheXagent-2-3b", "quantize": False,
     "load": load_chexagent_2_3b, "generate": generate_chexagent_2_3b},
    {"name": "chexagent_8b", "model_id": "StanfordAIMI/CheXagent-8b", "quantize": True,
     "load": load_chexagent_8b, "generate": generate_chexagent_8b},
    {"name": "medgemma_4b_it", "model_id": "google/medgemma-4b-it", "quantize": True,
     "load": load_generic_itt, "generate": generate_generic_itt},
    {"name": "qwen25_vl_3b", "model_id": "Qwen/Qwen2.5-VL-3B-Instruct", "quantize": True,
     "load": load_qwen25vl, "generate": generate_qwen25vl},
    {"name": "qwen25_vl_7b", "model_id": "Qwen/Qwen2.5-VL-7B-Instruct", "quantize": True,
     "load": load_qwen25vl, "generate": generate_qwen25vl},
    {"name": "phi35_vision", "model_id": "microsoft/Phi-3.5-vision-instruct", "quantize": True,
     "load": load_phi35_vision, "generate": generate_phi35_vision},
    {"name": "smolvlm2_2_2b", "model_id": "HuggingFaceTB/SmolVLM2-2.2B-Instruct", "quantize": True,
     "load": load_generic_itt, "generate": generate_generic_itt},
]
print(f"roster: {len(ROSTER)} models x {len(DATASETS)} datasets = "
      f"{len(ROSTER) * sum(len(d['manifest']) for d in DATASETS.values())} inferences")


In [ ]:
def run_one_dataset(generate_fn: Callable, model, proc_or_tok, dataset_name: str,
                    manifest: list[dict], crops_dir: str, out_path: Path) -> dict:
    """Run PROMPT over every crop in one dataset -> predictions_<model>_<dataset>.jsonl.

    A single image's failure is caught and recorded as an empty description rather than
    killing the model's whole run. The row is still written, so the scorer sees a prediction
    for every manifest id and n_missing_predictions stays 0.
    """
    n_ok = n_failed = 0
    t0 = time.time()
    with open(out_path, "w", encoding="utf-8") as f:
        for row in tqdm(manifest, desc=f"{dataset_name}", leave=False):
            image_path = f"{crops_dir}/{row['id']}.png"
            try:
                desc = generate_fn(proc_or_tok, model, image_path, PROMPT)
                n_ok += 1
            except Exception as e:
                desc = ""
                n_failed += 1
                if n_failed <= 3:      # cap the noise; the count is in the summary
                    print(f"  IMAGE FAILED {row['id']}: {e!r}")
            f.write(json.dumps({"id": row["id"], "kind": row["kind"],
                                 "description": desc}) + "\n")
    elapsed = time.time() - t0
    n_total = n_ok + n_failed
    rate = n_total / elapsed if elapsed > 0 else 0.0
    print(f"  {dataset_name}: {n_ok}/{n_total} ok | {elapsed:.1f}s | {rate:.2f} img/s | "
          f"{elapsed / max(n_total, 1):.2f} s/img -> {out_path.name}")
    return {"dataset": dataset_name, "n_ok": n_ok, "n_failed": n_failed,
            "elapsed_s": round(elapsed, 1), "images_per_sec": round(rate, 3)}


## 7. Benchmark Loop

One model at a time: load -> V4a -> V4b -> unload -> next. A model that fails (OOM, gated,
unsupported architecture) is logged and skipped; the run never stops. **No metrics here** --
this notebook ends at `predictions_*.jsonl`; scoring is local, via the project's existing
tested matchers.

In [ ]:
OUT_DIR = WORK_DIR / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

run_log: list[dict] = []
overall_t0 = time.time()

for spec in ROSTER:
    name, model_id = spec["name"], spec["model_id"]
    print(f"\n{'=' * 72}\n{name}  ({model_id})\n{'=' * 72}")
    vram_report(f"before {name}")

    model = proc_or_tok = None
    try:
        proc_or_tok, model = spec["load"](model_id, spec["quantize"])
        shard = getattr(model, "hf_device_map", None)
        print(f"  loaded ok" + (f" | sharded across {len(set(shard.values()))} device(s)"
                                 if shard else ""))
    except Exception as e:
        reason = f"{type(e).__name__}: {e}"
        print(f"  FAILED to load -- {reason}")
        traceback.print_exc()
        run_log.append({"model": name, "model_id": model_id, "status": "FAILED_LOAD",
                         "reason": reason})
        unload(model, proc_or_tok)
        vram_report(f"after {name}")
        continue

    model_results = []
    try:
        for dataset_name, d in DATASETS.items():
            out_path = OUT_DIR / f"predictions_{name}_{dataset_name}.jsonl"
            model_results.append(
                run_one_dataset(spec["generate"], model, proc_or_tok, dataset_name,
                                 d["manifest"], d["crops_dir"], out_path)
            )
        run_log.append({"model": name, "model_id": model_id, "status": "OK",
                         "results": model_results})
    except Exception as e:
        reason = f"{type(e).__name__}: {e}"
        print(f"  FAILED during inference -- {reason}")
        traceback.print_exc()
        run_log.append({"model": name, "model_id": model_id, "status": "FAILED_INFERENCE",
                         "reason": reason, "results": model_results})
    finally:
        unload(model, proc_or_tok)
        vram_report(f"after {name}")

print(f"\nTOTAL benchmark runtime: {time.time() - overall_t0:.1f}s")


## 8. Results

In [ ]:
n_ok_models = sum(1 for e in run_log if e["status"] == "OK")
print(f"{n_ok_models}/{len(ROSTER)} models produced predictions\n")
for entry in run_log:
    print(f"{entry['model']:18s} {entry['status']}")
    if entry["status"] == "OK":
        for r in entry["results"]:
            print(f"     {r['dataset']}: {r['n_ok']} ok / {r['n_failed']} failed | "
                  f"{r['images_per_sec']:.2f} img/s")
    else:
        print(f"     reason: {entry['reason'][:140]}")


## 9. Summary

Writes `run_log.md` / `run_log.json` next to the predictions, then zips everything to `/kaggle/working/outputs.zip`. Download from the Output tab in the right sidebar.

In [ ]:
lines = ["# Open-weight VLM roster benchmark -- run log\n\n"]
lines.append(f"- GPUs: {N_GPUS} x {torch.cuda.get_device_properties(0).name}\n")
lines.append(f"- dtype: {DTYPE} | transformers: {transformers.__version__}\n")
lines.append(f"- models OK: {n_ok_models}/{len(ROSTER)}\n\n")
for entry in run_log:
    lines.append(f"## {entry['model']} (`{entry['model_id']}`)\n\n")
    lines.append(f"Status: **{entry['status']}**\n\n")
    if entry["status"] == "OK":
        for r in entry["results"]:
            lines.append(f"- {r['dataset']}: {r['n_ok']}/{r['n_ok'] + r['n_failed']} ok, "
                          f"{r['elapsed_s']}s, {r['images_per_sec']} img/s\n")
    else:
        lines.append(f"- reason: `{entry['reason']}`\n")
    lines.append("\n")

(OUT_DIR / "run_log.md").write_text("".join(lines), encoding="utf-8")
(OUT_DIR / "run_log.json").write_text(json.dumps(run_log, indent=2), encoding="utf-8")

import shutil
shutil.make_archive(str(WORK_DIR / "outputs"), "zip", OUT_DIR)
print("".join(lines))
print(f"\nwrote {WORK_DIR / 'outputs.zip'} -- download via the Output tab (right sidebar)")
